In [2]:
import ee
import geemap
ee.Authenticate()  
ee.Initialize(project='...') 

# showcase the whole region 

In [15]:
# --- 1.1 Define the Area ---
alg_gaul = ee.FeatureCollection("FAO/GAUL/2015/level0")
alg = alg_gaul.filter(ee.Filter.eq("ADM0_NAME", "Algeria"))
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")
wilayas = ["Jijel", "Tizi Ouzou", "Bejaia", "Skikda", "Annaba", "El-Tarf", "Mila"]
target = gaul.filter( ee.Filter.inList("ADM1_NAME", wilayas) )

# Images
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(target)
    .filterDate("2026-09-01", "2026-09-10")
    .filterMetadata("CLOUDY_PIXEL_PERCENTAGE", "less_than", 1) )
print(f"Images found: {collection.size().getInfo()}")
# Sentinel-2 median composite
s2 = (collection.median().select(['B2', 'B3', 'B4']).clip(target))
# Wilaya boundaries
wilaya_borders = ee.Image().byte().paint( featureCollection = target, color=1,  width=2 )
Map = geemap.Map()
Map.addLayer(alg)
Map.centerObject(target, 8)
vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000,'gamma': 1.4 }
Map.addLayer(s2, vis_params)
Map.addLayer( wilaya_borders, {"palette": ["red"]}, "Wilaya Borders" )

Map

Images found: 15


Map(center=[36.64307804372882, 6.153246432011668], controls=(WidgetControl(options=['position', 'transparent_b…

# Study erea : Jijel BEFORE and AFTER the wildfires 

In [5]:
#  Define Jijel Wilaya
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")
target = gaul.filter( ee.Filter.eq("ADM1_NAME", "Jijel") )

In [6]:
# Get Sentinel-2 images for each period

# PRE-FIRE
pre_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(target)
    .filterDate("2026-08-12", "2026-08-22")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))  )

# POST-FIRE
post_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(target)
    .filterDate("2026-09-01", "2026-09-10")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))  )

print("Pre-fire images:", pre_collection.size().getInfo())
print("Post-fire images:", post_collection.size().getInfo())

Pre-fire images: 7
Post-fire images: 6


In [7]:
# Create median composites
bands = [
    "B2",   # Blue
    "B3",   # Green
    "B4",   # Red
    "B8",   # NIR
    "B11",  # SWIR 1
    "B12"   # SWIR 2
]

pre =  (pre_collection.median().select(bands).divide(10000).clip(target)   )
post = (post_collection.median().select(bands).divide(10000).clip(target)  )

#  Calculate NDVI and NBR 
ndvi_pre = ( pre.normalizedDifference(["B8", "B4"]).rename("NDVI_pre") )
ndvi_post = ( post.normalizedDifference(["B8", "B4"]).rename("NDVI_post") )
ndvi_diff = ( ndvi_post.subtract(ndvi_pre).rename("NDVI_difference") ) # Positive = NDVI increased   Negative = NDVI decreased

nbr_pre = (pre.normalizedDifference(["B8", "B12"]).rename("NBR_pre") )
nbr_post = (post.normalizedDifference(["B8", "B12"]).rename("NBR_post") )
nbr_diff = ( nbr_post.subtract(nbr_pre).rename("NBR_difference") )

In [8]:
# 5. ESA WorldCover forest mask
land_cover = ( ee.ImageCollection("ESA/WorldCover/v200").filterBounds(target).first().clip(target)  )
# Class 10 = Tree cover
forest_mask = (land_cover.eq(10).rename("forest") )
forest_mask_float = ( forest_mask.toFloat().rename("forest_mask") )

In [9]:
#  Define map
Map = geemap.Map()
Map.centerObject(target, 9)
# Jijel boundary
Map.addLayer( target,{"color": "red"},"Jijel Boundary" )
# RGB visualization
rgb_vis = { "bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3, "gamma": 1.4 }  
Map.addLayer( pre, rgb_vis,"Jijel - Before Fire" )
Map.addLayer( post, rgb_vis, "Jijel - After Fire" )

# NDVI visualization
ndvi_vis = { "min": -0.2,  "max": 0.9,  "palette": [ "brown", "yellow",  "lightgreen", "green", "darkgreen" ] }
Map.addLayer(  ndvi_pre, ndvi_vis,"NDVI - Before" )
Map.addLayer( ndvi_post, ndvi_vis, "NDVI - After" )

diff_vis = { "min": -0.6, "max": 0, "palette": [     "red",     "orange",     "white" ]  }
Map.addLayer( ndvi_diff, diff_vis, "NDVI Difference (After - Before)" )

# NBR
nbr_vis = {"min": -0.5, "max": 0.8, "palette": [     "brown",     "yellow",     "green",     "darkgreen" ] }
Map.addLayer(  nbr_pre, nbr_vis,"NBR - Before" )
Map.addLayer( nbr_post, nbr_vis, "NBR - After" )

nbr_diff_vis = {"min": -0.8, "max": 0,  "palette": [     "red",  "orange",    "white"  ] }
Map.addLayer( nbr_diff, nbr_diff_vis,"NBR Difference (After - Before)" )

# world cover 2021
forest_vis ={ "min": 0, "max": 1,   "palette": ["white", "darkgreen"] }
Map.addLayer( forest_mask, forest_vis , "Forest Mask 2021" )

Map

Map(center=[36.70310365863328, 5.9726545082825115], controls=(WidgetControl(options=['position', 'transparent_…

In [12]:
# Calculate total Jijel area

jijel_area = (target.geometry().area(1))
jijel_area_km2 = jijel_area.divide(1e6)
print(f"Jijel area (km²): {jijel_area_km2.getInfo():.3f}")

# 12. Calculate mean NDVI before and after
ndvi_pre_mean  = ( ndvi_pre.reduceRegion ( reducer=ee.Reducer.mean(), geometry=target.geometry(),  scale=10,  maxPixels=1e10 ).get("NDVI_pre")  )
ndvi_post_mean = ( ndvi_post.reduceRegion( reducer=ee.Reducer.mean(), geometry=target.geometry(),  scale=10,  maxPixels=1e10 ).get("NDVI_post") )
print(f"Mean NDVI before: {ndvi_pre_mean.getInfo():.3f}")
print(f"Mean NDVI after:  {ndvi_post_mean.getInfo():.3f}")

# 13. Mean NDVI difference
mean_ndvi_diff = ( ndvi_diff.reduceRegion(  reducer=ee.Reducer.mean(),  geometry=target.geometry(),  scale=10,  maxPixels=1e10).get("NDVI_difference")  )
print(f"Mean NDVI difference: {mean_ndvi_diff.getInfo():.3f}")

# 14. Significant NDVI decrease
ndvi_loss_mask = ndvi_diff.lt(-0.2)
# Pixel area in km²
pixel_area_km2 = ee.Image.pixelArea().divide(1e6)
ndvi_loss_area = (
    pixel_area_km2
    .updateMask(ndvi_loss_mask)
    .reduceRegion( reducer=ee.Reducer.sum(), geometry=target.geometry(),scale=10, maxPixels=1e10 )
    .get("area") )

ndvi_loss_area_km2 = ee.Number(ndvi_loss_area)
print(f"Area with NDVI decrease < -0.2 (km²): "  f"{ndvi_loss_area_km2.getInfo():.3f}" ) # Percentage of Jijel
ndvi_loss_percentage = ( ndvi_loss_area_km2.divide(jijel_area_km2) .multiply(100) )
print(f"Percentage of Jijel with NDVI decrease (< -0.2): " f"{ndvi_loss_percentage.getInfo():.3f}%")

Jijel area (km²): 2382.599
Mean NDVI before: 0.521
Mean NDVI after:  0.399
Mean NDVI difference: -0.122
Area with NDVI decrease < -0.2 (km²): 570.246
Percentage of Jijel with NDVI decrease (< -0.2): 23.934%


In [13]:
# 15. Calculate mean NBR before and after
nbr_pre_mean = (nbr_pre.reduceRegion(reducer=ee.Reducer.mean(),geometry=target.geometry(),scale=10,maxPixels=1e10).get("NBR_pre") )
nbr_post_mean = ( nbr_post.reduceRegion( reducer=ee.Reducer.mean(), geometry=target.geometry(),  scale=10,  maxPixels=1e10 ) .get("NBR_post") )
print(f"Mean NBR before: {nbr_pre_mean.getInfo():.3f}")
print(f"Mean NBR after: {nbr_post_mean.getInfo():.3f}")
# 16. Mean NBR difference

mean_nbr_diff = (nbr_diff.reduceRegion(reducer=ee.Reducer.mean(),geometry=target.geometry(),scale=10,maxPixels=1e10) .get("NBR_difference"))
print(f"Mean NBR difference: {mean_nbr_diff.getInfo():.3f}")

# 17. Significant NBR decrease
nbr_loss_mask = nbr_diff.lt(-0.2)
# Pixel area in km²
pixel_area_km2 = ee.Image.pixelArea().divide(1e6)
nbr_loss_area = (
    pixel_area_km2
    .updateMask(nbr_loss_mask)
    .reduceRegion( reducer=ee.Reducer.sum(), geometry=target.geometry(), scale=10, maxPixels=1e10 )
    .get("area") )

nbr_loss_area_km2 = ee.Number(nbr_loss_area) 
print(f"Area with NBR decrease < -0.2 (km²): {nbr_loss_area_km2.getInfo():.3f}")

# 18. Percentage of Jijel
nbr_loss_percentage = (nbr_loss_area_km2.divide(jijel_area_km2).multiply(100))
print(f"Percentage of Jijel with NBR decrease (< -0.2): {nbr_loss_percentage.getInfo():.3f}%")

Mean NBR before: 0.312
Mean NBR after: 0.109
Mean NBR difference: -0.203
Area with NBR decrease < -0.2 (km²): 663.275
Percentage of Jijel with NBR decrease (< -0.2): 27.838%


# Installation 

In [8]:
# 10. Build the two export images

pre_export = (
    pre
    .addBands(ndvi_pre)
    .addBands(nbr_pre)
    .addBands(forest_mask_float)
    .toFloat()
)

post_export = (
    post
    .addBands(ndvi_post)
    .addBands(nbr_post)
    .addBands(forest_mask_float)
    .toFloat()
)


print("Pre-fire bands:", pre_export.bandNames().getInfo())
print("Post-fire bands:", post_export.bandNames().getInfo())


# 11. Export PRE-FIRE TIFF

task_pre = ee.batch.Export.image.toDrive(
    image=pre_export,
    description="jijel_pre_fire",
    folder="GEE_Exports",
    fileNamePrefix="jijel_pre_fire",
    region=target.geometry(),
    scale=10,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

task_pre.start()

# 12. Export POST-FIRE TIFF

task_post = ee.batch.Export.image.toDrive(
    image=post_export,
    description="jijel_post_fire",
    folder="GEE_Exports",
    fileNamePrefix="jijel_post_fire",
    region=target.geometry(),
    scale=10,
    fileFormat="GeoTIFF",
    maxPixels=1e13 )

task_post.start()

print("Exports started.")

Pre-fire bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI_pre', 'NBR_pre', 'forest_mask']
Post-fire bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI_post', 'NBR_post', 'forest_mask']
Exports started.
